# AI Banking Impact Research

# Lab Problem Statement: AI-Powered Insurance Claim Analysis using CrewAI

## Background

An insurance company receives hundreds of customer claim requests every day through emails and web forms. Claims include information such as:

- Accident details
- Policy number
- Claim amount
- Supporting descriptions

Manually reviewing these claims is time-consuming and can delay customer settlements.

The company wants to build an **AI Multi-Agent System** that can automatically analyze incoming insurance claims, identify important information, assess potential risks, and generate a structured report for claim officers.

---

# Problem Statement

Develop a **CrewAI-based Multi-Agent System** that analyzes an insurance claim and produces a comprehensive claim assessment report.

The system should:

1. Read an insurance claim provided by the user.
2. Extract key claim information:
   - Policy number
   - Claim type
   - Claim amount
   - Incident description
   - Date
   - Other relevant details
3. Identify possible fraud indicators or missing information.
4. Assess the overall risk level:
   - Low
   - Medium
   - High
5. Generate a concise summary for the insurance claim officer.
6. Recommend whether the claim should be:
   - ✅ Approved
   - ⚠️ Sent for Manual Review
   - ❌ Rejected (with justification)

---

# Expected Output

The AI system should generate a report containing:

- Claim Summary
- Extracted Key Information
- Missing or Incomplete Details
- Potential Fraud Indicators
- Risk Assessment
- Recommendation
- Reasoning Behind the Recommendation

---

# Learning Objectives

This lab helps learners understand:

- Multi-Agent Systems using **CrewAI**
- Agent collaboration
- Task orchestration
- Structured report generation using LLMs
- Real-world enterprise AI workflows without requiring external APIs or complex integrations

In [1]:
import os

# 🔑 Add your Groq API Key here
#os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [1]:
from crewai import LLM

llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.2,
    api_key=os.getenv("GROQ_API_KEY")
)


NameError: name 'os' is not defined

In [6]:
insurance_claim = """
    Policy Number: POL-2026-45891
    Customer Name: Rahul Sharma
    Claim Type: Motor Accident
    Claim Amount: INR 2,50,000
    Incident Date: 12 July 2026
    Claim Submission Date: 17 July 2026
    Incident Description:
    The customer reported that his car collided with a divider while
    returning home at approximately 11:30 PM. The front part of the car
    was severely damaged.
    Additional Information:
    - Police report is not attached.
    - Hospital records are not applicable.
    - Vehicle repair estimate is INR 2,50,000.
    - Images of the damaged vehicle are attached.
    - The policy was purchased 25 days before the accident.
    - The customer submitted two motor claims during the previous year.
"""

In [7]:
from crewai import Agent

claim_analyst_agent = Agent(
    role="Insurance Claim Analyst",
    goal=(
        "Extract and verify all important information from an insurance "
        "claim and identify incomplete or inconsistent information."
    ),
    backstory=(
        "You are an experienced insurance claim analyst with expertise "
        "in motor, health and property insurance claims"
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

fraud_risk_analyst_agent = Agent(
    role="Insurance Fraud and Risk Analyst",
    goal=(
        "Identify fraud indicators, assess claim risk and recommend the "
        "appropriate action."
    ),
    backstory=(
        "You specialize in insurance fraud detection, risk assessment "
        "and responsible claim decision-making."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

report_writer_agent = Agent(
    role="Insurance Claim Report Writer",
    goal=(
        "Prepare a clear and strucutred claim assessment report for  "
        "an insurance claim officer."
    ),
    backstory=(
        "You prepare concise business reports that help insurance officers "
        "make informed decisions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)


## Define Task

In [8]:
from crewai import Task

claim_analyst_task = Task(
    description=f""" 
        Analyze the following insurance claim:
        {insurance_claim}
        Extract:
        1. Policy number
        2. Customer name
        3. Claim type
        4. Claim amount
        5. Incident date
        6. Submission date
        7. Incident description
        8. Supporting documents
        9. Missing information
        10. Inconsistent or suspicious information
        Do not invent any information that is not present in the claim.
    """,
    expected_output=(
        "A strucutred claim analysis containing extracted details, " 
        "available documents, missing details and inconsistencies."
        ),
    agent=claim_analyst_agent
)

fraud_risk_analyst_task = Task(
    description="""
        Review the extracted insurance claim information.
        Perform the following analysis:
        1. Identify possible fraud indicators.
        2. Assess the claim risk as Low, Medium or High.
        3. Explain the reason for the selected risk level.
        4. Recommend one of these actions:
        - Approve
        - Send for Manual Review
        - Reject
        5. Avoid making a final rejection solely based on missing documents.
        6. Consider fairness, transparency and responsible-AI principles.
        """,
    expected_output=(
        "A risk assessment containing fraud indicators, risk level, "
        "reasoning and recommended action."
    ),
    agent=fraud_risk_analyst_agent,
    context=[claim_analyst_task]
)


report_writer_task = Task(
    description="""
        Using the claim analysis and risk assessment, prepare a final report.
        Use the following sections:
        # Insurance Claim Assessment Report
        ## 1. Claim Summary
        ## 2. Extracted Key Information
        ## 3. Available Supporting Documents
        ## 4. Missing or Incomplete Information
        ## 5. Potential Fraud Indicators
        ## 6. Risk Assessment
        ## 7. Recommendation
        ## 8. Reasoning
        ## 9. Responsible-AI Considerations
        ## 10. Required Next Steps
        The report must be professional, concise and easy for a claim officer
        to understand.
        Clearly state that the AI-generated recommendation must be reviewed by
        an authorized human claim officer.
    """,
    expected_output=(
        "A complete Markdown insurance claim assessment report with all "
        "requested sections."
    ),
    agent=report_writer_agent,
    context=[claim_analyst_task, fraud_risk_analyst_task]
)


## Create Crew

In [10]:
from crewai import Crew, Process

crew = Crew(
    agents=[claim_analyst_agent, fraud_risk_analyst_agent, report_writer_agent],
    tasks=[claim_analyst_task, fraud_risk_analyst_task, report_writer_task],
    process=Process.sequential,
    tracing=False,
    verbose=True
)


## 7. Execute the crew

In [11]:
result = await crew.kickoff_async()
print("\n" + "=" * 80)
print("FINAL CLAIM ASSESSMENT")
print("=" * 80)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5d98dec0-59e9-497f-bd0f-fea5fd9a2581                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze the following insurance claim:                                                                 │
│                                                                                                                 │
│      Policy Number: POL-2026-45891                                                                              │
│      Customer Name: Rahul Sharma                                                                                │
│      Claim Type: Motor Accident                                                                                 │
│      Claim Amount: INR 2,50,000                                                                                 │
│      Incident Date: 12 July 2026                                                                                │
│      Claim Submission Date: 17 July 2026                                                                        │
│      Incident Description:                                                                                      │
│      The customer reported that his car collided with a divider while                                           │
│      returning home at approximately 11:30 PM. The front part of the car                                        │
│      was severely damaged.                                                                                      │
│      Additional Information:                                                                                    │
│      - Police report is not attached.                                                                           │
│      - Hospital records are not applicable.                                                                     │
│      - Vehicle repair estimate is INR 2,50,000.                                                                 │
│      - Images of the damaged vehicle are attached.                                                              │
│      - The policy was purchased 25 days before the accident.                                                    │
│      - The customer submitted two motor claims during the previous year.                                        │
│                                                                                                                 │
│          Extract:                                                                                               │
│          1. Policy number                                                                                       │
│          2. Customer name                                                                                       │
│          3. Claim type                                                                                          │
│          4. Claim amount                                                                                        │
│          5. Incident date                                                                                       │
│          6. Submission date                                                                                     │
│          7. Incident description                                                                                │
│          8. Supporting documents                                                                                │
│          9. Missing information                                                                                 │
│          10. Inconsistent or suspicious information    

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Insurance Claim Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the following insurance claim:                                                                 │
│                                                                                                                 │
│      Policy Number: POL-2026-45891                                                                              │
│      Customer Name: Rahul Sharma                                                                                │
│      Claim Type: Motor Accident                                                                                 │
│      Claim Amount: INR 2,50,000                                                                                 │
│      Incident Date: 12 July 2026                                                                                │
│      Claim Submission Date: 17 July 2026                                                                        │
│      Incident Description:                                                                                      │
│      The customer reported that his car collided with a divider while                                           │
│      returning home at approximately 11:30 PM. The front part of the car                                        │
│      was severely damaged.                                                                                      │
│      Additional Information:                                                                                    │
│      - Police report is not attached.                                                                           │
│      - Hospital records are not applicable.                                                                     │
│      - Vehicle repair estimate is INR 2,50,000.                                                                 │
│      - Images of the damaged vehicle are attached.                                                              │
│      - The policy was purchased 25 days before the accident.                                                    │
│      - The customer submitted two motor claims during the previous year.                                        │
│                                                                                                                 │
│          Extract:                                                                                               │
│          1. Policy number                                                                                       │
│          2. Customer name                                                                                       │
│          3. Claim type                                                                                          │
│          4. Claim amount                                                                                        │
│          5. Incident date                                                                                       │
│          6. Submission date                                                                                     │
│          7. Incident description                                                                                │
│          8. Supporting documents                                                                                │
│          9. Missing information                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Insurance Claim Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Structured Claim Analysis**                                                                                  │
│                                                                                                                 │
│  **1. Extracted Details:**                                                                                      │
│                                                                                                                 │
│  - Policy Number: POL-2026-45891                                                                                │
│  - Customer Name: Rahul Sharma                                                                                  │
│  - Claim Type: Motor Accident                                                                                   │
│  - Claim Amount: INR 2,50,000                                                                                   │
│  - Incident Date: 12 July 2026                                                                                  │
│  - Claim Submission Date: 17 July 2026                                                                          │
│  - Incident Description: The customer reported that his car collided with a divider while returning home at     │
│  approximately 11:30 PM. The front part of the car was severely damaged.                                        │
│  - Additional Information:                                                                                      │
│    - The policy was purchased 25 days before the accident.                                                      │
│    - The customer submitted two motor claims during the previous year.                                          │
│                                                                                                                 │
│  **2. Supporting Documents:**                                                                                   │
│                                                                                                                 │
│  - Images of the damaged vehicle are attached.                                                                  │
│                                                                                                                 │
│  **3. Missing Information:**                                                                                    │
│                                                                                                                 │
│  - Police report is not attached.                                                                               │
│  - Hospital records are not applicable (as this is a motor accident claim, hospital records are not expected,   │
│  but it's worth noting for completeness).                                                                       │
│                                                                                                                 │
│  **4. Inconsistent or Suspicious Information:**                                                                 │
│                                                                                                                 │
│  - The claim amount is equal to the vehicle repair estimate, which may raise questions about the authenticity   │
│  of the claim.                                         

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Insurance Claim Report Writer                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Insurance Claim Assessment Report                                                                            │
│                                                                                                                 │
│  ## 1. Claim Summary                                                                                            │
│                                                                                                                 │
│  * Policy Number: POL-2026-45891                                                                                │
│  * Customer Name: Rahul Sharma                                                                                  │
│  * Claim Type: Motor Accident                                                                                   │
│  * Claim Amount: INR 2,50,000                                                                                   │
│  * Incident Date: 12 July 2026                                                                                  │
│  * Claim Submission Date: 17 July 2026                                                                          │
│                                                                                                                 │
│  ## 2. Extracted Key Information                                                                                │
│                                                                                                                 │
│  * The policy was purchased 25 days before the accident.                                                        │
│  * The customer submitted two motor claims during the previous year.                                            │
│  * The claim amount is equal to the vehicle repair estimate.                                                    │
│  * The incident occurred at night, which may have contributed to the severity of the damage.                    │
│  * The claim was submitted within 5 days of the incident.                                                       │
│                                                                                                                 │
│  ## 3. Available Supporting Documents                                                                           │
│                                                                                                                 │
│  * Images of the damaged vehicle are attached.                                                                  │
│                                                                                                                 │
│  ## 4. Missing or Incomplete Information                                                                        │
│                                                                                                                 │
│  * Police report is not attached.                                                                               │
│  * Hospital records are not applicable (as this is a motor accident claim, hospital records are not expected,   │
│  but it's worth noting for completeness).                                                                       │
│                                                                                                                 │
│  ## 5. Potential Fraud Indicators                      

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Using the claim analysis and risk assessment, prepare a final report.                                  │
│          Use the following sections:                                                                            │
│          # Insurance Claim Assessment Report                                                                    │
│          ## 1. Claim Summary                                                                                    │
│          ## 2. Extracted Key Information                                                                        │
│          ## 3. Available Supporting Documents                                                                   │
│          ## 4. Missing or Incomplete Information                                                                │
│          ## 5. Potential Fraud Indicators                                                                       │
│          ## 6. Risk Assessment                                                                                  │
│          ## 7. Recommendation                                                                                   │
│          ## 8. Reasoning                                                                                        │
│          ## 9. Responsible-AI Considerations                                                                    │
│          ## 10. Required Next Steps                                                                             │
│          The report must be professional, concise and easy for a claim officer                                  │
│          to understand.                                                                                         │
│          Clearly state that the AI-generated recommendation must be reviewed by                                 │
│          an authorized human claim officer.                                                                     │
│                                                                                                                 │
│  Agent: Insurance Claim Report Writer                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL CLAIM ASSESSMENT
# Insurance Claim Assessment Report

## 1. Claim Summary

* Policy Number: POL-2026-45891
* Customer Name: Rahul Sharma
* Claim Type: Motor Accident
* Claim Amount: INR 2,50,000
* Incident Date: 12 July 2026
* Claim Submission Date: 17 July 2026

## 2. Extracted Key Information

* The policy was purchased 25 days before the accident.
* The customer submitted two motor claims during the previous year.
* The claim amount is equal to the vehicle repair estimate.
* The incident occurred at night, which may have contributed to the severity of the damage.
* The claim was submitted within 5 days of the incident.

## 3. Available Supporting Documents

* Images of the damaged vehicle are attached.

## 4. Missing or Incomplete Information

* Police report is not attached.
* Hospital records are not applicable (as this is a motor accident claim, hospital records are not expected, but it's worth noting for completeness).

## 5. Potential Fraud Indicators

* Inconsistent Pol

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5d98dec0-59e9-497f-bd0f-fea5fd9a2581                                                                       │
│  Final Output: # Insurance Claim Assessment Report                                                              │
│                                                                                                                 │
│  ## 1. Claim Summary                                                                                            │
│                                                                                                                 │
│  * Policy Number: POL-2026-45891                                                                                │
│  * Customer Name: Rahul Sharma                                                                                  │
│  * Claim Type: Motor Accident                                                                                   │
│  * Claim Amount: INR 2,50,000                                                                                   │
│  * Incident Date: 12 July 2026                                                                                  │
│  * Claim Submission Date: 17 July 2026                                                                          │
│                                                                                                                 │
│  ## 2. Extracted Key Information                                                                                │
│                                                                                                                 │
│  * The policy was purchased 25 days before the accident.                                                        │
│  * The customer submitted two motor claims during the previous year.                                            │
│  * The claim amount is equal to the vehicle repair estimate.                                                    │
│  * The incident occurred at night, which may have contributed to the severity of the damage.                    │
│  * The claim was submitted within 5 days of the incident.                                                       │
│                                                                                                                 │
│  ## 3. Available Supporting Documents                                                                           │
│                                                                                                                 │
│  * Images of the damaged vehicle are attached.                                                                  │
│                                                                                                                 │
│  ## 4. Missing or Incomplete Information                                                                        │
│                                                                                                                 │
│  * Police report is not attached.                                                                               │
│  * Hospital records are not applicable (as this is a motor accident claim, hospital records are not expected,   │
│  but it's worth noting for completeness).                                                                       │
│                                                                                                                 │
│  ## 5. Potential Fraud Indicators                     

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Save the final report

In [15]:
OUTPUT = "crewai/content"

import os
os.makedirs(OUTPUT, exist_ok=True)

output_file = f"{OUTPUT}/insurance_claim_assessment.md"

with open(output_file, "w", encoding="utf-8") as file:
    file.write(result.raw)

print(f"Report saved successfully: {output_file}")

Report saved successfully: crewai/content/insurance_claim_assessment.md
